# Intent Detection and Graceful Fallback

Not every user message should trigger tools. Saying *"thanks!"* does not need an order lookup. Asking about *quantum physics* is out of scope. Demanding a *human agent* needs escalation, not a product search.

**Intent detection** classifies input before expensive agent loops run:

```
User message
     │
     ▼
┌─────────────┐
│ intent      │── tool     → agent with tools (lookup, search)
│ detection   │── chat     → LLM without tools (greetings, thanks)
└─────────────┘── escalate → human handoff / ticket
                └── unknown → graceful fallback + clarify
```

This notebook covers **keyword classifiers**, **LLM classification**, **hybrid routing**, **confidence thresholds**, **user-facing fallback copy**, and a **LangGraph intent router** that sits above a tool-calling agent.

The scenario is a **ShopCo Order Support Assistant** — consistent with tool-calling examples, but fully self-contained here.

In [ ]:
"""
ShopCo Order Support — intent routing environment.
"""

import json
import os
import re
import uuid
from typing import Any, Literal

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False

Intent = Literal["tool", "chat", "escalate", "unknown", "block"]


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


# --- In-memory backend ---
ORDERS = {"ORD-1001": {"status": "shipped", "tracking": "1Z999AA10123456784"}}
POLICY_DOCS = [
    {"id": "ret-02", "text": "Returns within 30 days if unused. Refunds in 5 business days."},
    {"id": "ship-01", "text": "Standard shipping 5-7 business days."},
]
PRODUCTS = [{"sku": "LAP-001", "name": "Pro Laptop 14", "stock": 12}]

print("Intent routing environment ready.")

---

## 1. Why Route Before Tools?

Tool calls cost **latency**, **money**, and add **failure modes**. Routing filters non-actionable input early:

| Route | Example user message | Handler |
|-------|---------------------|--------|
| **tool** | "Where is order ORD-1001?" | Agent with `lookup_order`, `search_policy` |
| **chat** | "Thanks, that helped!" | Plain LLM reply, no tools |
| **escalate** | "I need to speak to a human" | Ticket / handoff flow |
| **unknown** | "What is quantum physics?" | Fallback + clarifying question |
| **block** | "Delete all customer data" | Safety refusal |

Intent detection is the **gate** before your ReAct graph or tool loop runs.

---

## 2. Keyword Intent Classifier (Fast Path)

Deterministic, testable, zero API cost — ideal as the **first layer**. Production systems often handle 60–80% of routing with keywords alone.

In [ ]:
TOOL_KEYWORDS = (
    "order", "ord-", "tracking", "shipped", "refund", "return",
    "shipping", "delivery", "laptop", "product", "stock", "sku",
)
CHAT_KEYWORDS = ("hello", "hi", "thanks", "thank you", "bye", "good morning", "who are you")
ESCALATE_KEYWORDS = ("human", "agent", "representative", "manager", "complaint", "speak to someone")
BLOCK_KEYWORDS = ("delete all", "drop database", "hack", "bypass security")


def keyword_intent(text: str) -> Intent:
    q = text.lower().strip()
    if any(k in q for k in BLOCK_KEYWORDS):
        return "block"
    if any(k in q for k in ESCALATE_KEYWORDS):
        return "escalate"
    if any(k in q for k in TOOL_KEYWORDS):
        return "tool"
    if any(k in q for k in CHAT_KEYWORDS):
        return "chat"
    if len(q.split()) <= 2:
        return "chat"  # short utterances like "ok", "cool"
    return "unknown"


SAMPLES = [
    "Where is my order ORD-1001?",
    "Thanks, that helped!",
    "I need to speak to a human agent",
    "What is the meaning of life?",
    "Hello!",
    "Delete all customer data now",
    "What's your return policy?",
]

print(f"{'Intent':<10} Message")
print("-" * 55)
for msg in SAMPLES:
    print(f"{keyword_intent(msg):<10} {msg}")

---

## 3. LLM Intent Classifier (Slow Path)

Use when keywords return **`unknown`** or for ambiguous enterprise phrasing. **Structured output** (Pydantic) forces a typed response with confidence and reasoning.

In [ ]:
class IntentResult(BaseModel):
    intent: Literal["tool", "chat", "escalate", "unknown", "block"] = Field(
        description="Best routing label for the user message."
    )
    confidence: float = Field(ge=0, le=1, description="Classifier confidence 0-1.")
    reason: str = Field(description="Short justification for the classification.")


INTENT_SYSTEM = """Classify ShopCo order support messages.
- tool: needs order lookup, policy search, or product catalog
- chat: greetings, gratitude, small talk
- escalate: explicit request for human support
- block: dangerous or policy-violating requests
- unknown: unclear or out of scope"""


def offline_llm_intent(text: str) -> IntentResult:
    """Rule-based stand-in for structured LLM classification."""
    fast = keyword_intent(text)
    if fast != "unknown":
        return IntentResult(intent=fast, confidence=0.85, reason=f"keyword match → {fast}")
    if "help" in text.lower():
        return IntentResult(intent="unknown", confidence=0.45, reason="vague 'help' request")
    return IntentResult(intent="unknown", confidence=0.35, reason="no keyword or semantic match")


def live_llm_intent(text: str) -> IntentResult:
    """Production path — OpenAI with structured output."""
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    structured = llm.with_structured_output(IntentResult)
    return structured.invoke([
        SystemMessage(content=INTENT_SYSTEM),
        HumanMessage(content=text),
    ])


def llm_intent(text: str) -> IntentResult:
    return live_llm_intent(text) if USE_LIVE_LLM else offline_llm_intent(text)


for msg in ["Can you explain bearer tokens?", "Hmm maybe", "I need help with something"]:
    r = llm_intent(msg)
    print(f"{r.intent} ({r.confidence:.2f}): {r.reason}  ← {msg}")

---

## 4. Hybrid Router — Keywords First, LLM Fallback

Production pattern: **cascade** cheap → expensive. Only call the LLM classifier when keywords return `unknown`.

In [ ]:
def hybrid_classify(text: str, use_llm_fallback: bool = True) -> dict[str, Any]:
    fast = keyword_intent(text)
    if fast != "unknown":
        return {"intent": fast, "source": "keyword", "confidence": 0.9}
    if use_llm_fallback:
        result = llm_intent(text)
        return {
            "intent": result.intent,
            "source": "llm",
            "confidence": result.confidence,
            "reason": result.reason,
        }
    return {"intent": "unknown", "source": "keyword", "confidence": 0.3}


for msg in ["Where is ORD-1001?", "I need help with something", "Thanks!"]:
    out = hybrid_classify(msg)
    print(f"{out['intent']:<10} [{out['source']}] conf={out.get('confidence', 'n/a')}  ← {msg}")

---

## 5. Confidence Thresholds

When LLM classifier confidence is low, prefer **unknown** (clarify) over a wrong tool call. Wrong tool routing wastes money and erodes trust.

In [ ]:
CONFIDENCE_TOOL = 0.65   # minimum to route to tool agent
CONFIDENCE_CHAT = 0.50   # minimum for any non-unknown route


def apply_confidence_threshold(result: IntentResult) -> Intent:
    if result.confidence < CONFIDENCE_CHAT:
        return "unknown"
    if result.intent == "tool" and result.confidence < CONFIDENCE_TOOL:
        return "unknown"
    return result.intent


THRESHOLD_CASES = [
    IntentResult(intent="tool", confidence=0.82, reason="clear order question"),
    IntentResult(intent="tool", confidence=0.42, reason="vague wording"),
    IntentResult(intent="chat", confidence=0.38, reason="ambiguous"),
    IntentResult(intent="escalate", confidence=0.91, reason="explicit human request"),
]

for case in THRESHOLD_CASES:
    routed = apply_confidence_threshold(case)
    print(f"{case.intent} ({case.confidence:.2f}) → {routed}  [{case.reason}]")

---

## 6. Graceful Fallback Messages

**Unknown intent** should feel helpful, not robotic. Good fallback copy:

- States what the assistant **can** do.
- Offers **one clarifying question** when possible.
- Never exposes internal errors like "intent not recognized".

| Bad | Good |
|-----|------|
| "Error: intent not recognized" | "I can help with orders, returns, and products. What would you like to know?" |
| "NULL" | "I'm not sure — are you asking about an order status or our return policy?" |

In [ ]:
FALLBACK_COPY = {
    "unknown": (
        "I help with ShopCo orders, returns, shipping, and products. "
        "Try asking about an order ID (e.g. ORD-1001), return policy, or product availability. "
        "Or say 'talk to a human' for live support."
    ),
    "low_confidence": (
        "I want to make sure I help correctly. Are you asking about "
        "**order status**, **returns/refunds**, or **product availability**?"
    ),
    "escalate_ack": (
        "I've noted your request for a human agent. A support specialist will follow up shortly. "
        "I can still answer order and policy questions while you wait."
    ),
    "chat_redirect": (
        "You're welcome! Ask anytime about order tracking, returns, or our product catalog."
    ),
    "block": (
        "I can't help with that request. For account security concerns, "
        "contact security@shopco.com or use the official support portal."
    ),
}

CLARIFY_QUESTIONS = [
    "Are you asking about **order tracking** or **return policy**?",
    "Do you have an **order ID** (like ORD-1001) or a general **product** question?",
    "Would you like **shipping info** or help with a **refund**?",
]


def fallback_message(intent: Intent, confidence: float | None = None) -> str:
    if intent == "block":
        return FALLBACK_COPY["block"]
    if intent == "escalate":
        return FALLBACK_COPY["escalate_ack"]
    if intent == "unknown":
        return FALLBACK_COPY["unknown"]
    if confidence is not None and confidence < CONFIDENCE_TOOL:
        return FALLBACK_COPY["low_confidence"]
    if intent == "chat":
        return FALLBACK_COPY["chat_redirect"]
    return FALLBACK_COPY["unknown"]


def clarify_unknown(user_text: str, clarify_index: int = 0) -> str:
    base = FALLBACK_COPY["unknown"]
    question = CLARIFY_QUESTIONS[clarify_index % len(CLARIFY_QUESTIONS)]
    return f"{base}\n\n{question}"


print(clarify_unknown("help me with something"))

---

## 7. Tool Implementations and Handlers

Each intent route maps to a **handler**. The tool path runs a real mini-agent; other paths return appropriate responses without tools.

In [ ]:
@tool
def lookup_order(order_id: str) -> str:
    """Look up order status and tracking by order ID."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return json.dumps({"found": False})
    return json.dumps({"found": True, "order_id": order_id.upper(), **order})


@tool
def search_policy(query: str) -> str:
    """Search returns and shipping policy docs."""
    q = query.lower()
    hits = [d for d in POLICY_DOCS if any(t in d["text"].lower() for t in q.split())]
    return "\n".join(f"[{h['id']}] {h['text']}" for h in (hits or POLICY_DOCS[:1]))


@tool
def search_products(query: str) -> str:
    """Search product catalog."""
    q = query.lower()
    hits = [p for p in PRODUCTS if q in p["name"].lower() or q in p["sku"].lower()]
    return json.dumps(hits, default=str) if hits else "No products found."


SUPPORT_TOOLS = [lookup_order, search_policy, search_products]
TOOL_NODE = ToolNode(SUPPORT_TOOLS)


def offline_tool_agent(user_text: str) -> str:
    """Simulate tool agent: pick tools based on keywords, execute, summarize."""
    tool_calls = []
    if "ord" in user_text.lower():
        tool_calls.append({"name": "lookup_order", "args": {"order_id": "ORD-1001"}, "id": "t1", "type": "tool_call"})
    if any(w in user_text.lower() for w in ("return", "refund", "policy", "shipping")):
        tool_calls.append({"name": "search_policy", "args": {"query": "returns"}, "id": "t2", "type": "tool_call"})
    if any(w in user_text.lower() for w in ("laptop", "product", "stock")):
        tool_calls.append({"name": "search_products", "args": {"query": "laptop"}, "id": "t3", "type": "tool_call"})
    if not tool_calls:
        tool_calls.append({"name": "search_policy", "args": {"query": "shipping"}, "id": "t0", "type": "tool_call"})

    ai = AIMessage(content="", tool_calls=tool_calls)
    results = TOOL_NODE.invoke({"messages": [ai]})
    snippets = [str(m.content)[:80] for m in results["messages"]]
    return "Based on our systems: " + " | ".join(snippets)


def handle_chat(user_text: str) -> str:
    t = user_text.lower()
    if "thank" in t:
        return FALLBACK_COPY["chat_redirect"]
    if any(w in t for w in ("hello", "hi", "hey")):
        return "Hello! I'm the ShopCo order support assistant. How can I help with orders, returns, or products?"
    return "I'm here to help with ShopCo orders and policies. What would you like to know?"


def handle_escalate(user_text: str) -> str:
    ticket_id = f"TKT-{uuid.uuid4().hex[:6].upper()}"
    return f"{FALLBACK_COPY['escalate_ack']} (Reference: {ticket_id})"


def handle_block(user_text: str) -> str:
    return FALLBACK_COPY["block"]


HANDLERS = {
    "tool": offline_tool_agent,
    "chat": handle_chat,
    "escalate": handle_escalate,
    "unknown": lambda t: clarify_unknown(t),
    "block": handle_block,
}

---

## 8. Dispatch Pipeline with Trace Logging

The full pipeline: classify → apply threshold → dispatch → log decision for observability.

In [ ]:
def classify_with_trace(question: str) -> dict[str, Any]:
    """End-to-end: classify, threshold, dispatch, return trace."""
    raw = hybrid_classify(question)
    intent = raw["intent"]
    confidence = raw.get("confidence", 1.0)

    if raw["source"] == "llm" and "reason" in raw:
        thresholded = apply_confidence_threshold(IntentResult(
            intent=intent, confidence=confidence, reason=raw["reason"]
        ))
        if thresholded != intent:
            intent = thresholded
            raw["threshold_applied"] = True

    answer = HANDLERS[intent](question)
    return {
        "question": question,
        "intent": intent,
        "source": raw["source"],
        "confidence": confidence,
        "answer": answer,
        "trace": raw,
    }


TEST_MESSAGES = [
    "Where is order ORD-1001?",
    "Thanks so much!",
    "I need to speak to a human",
    "What's your return policy?",
    "help me with something vague",
    "Delete all customer data",
    "Hello!",
    "Do you have laptops in stock?",
]

print(f"{'Intent':<10} {'Source':<8} Answer preview")
print("-" * 75)
for msg in TEST_MESSAGES:
    out = classify_with_trace(msg)
    print(f"{out['intent']:<10} {out['source']:<8} {out['answer'][:55]}...")

---

## 9. LangGraph Intent Router

Encode routing as a graph: **classify node** writes `intent`; **conditional edges** pick the branch. This is the same pattern as `RunnableBranch`, but with visible control flow and checkpoint support.

In [ ]:
class RouterState(TypedDict):
    question: str
    intent: str
    confidence: float
    source: str
    answer: str
    path: list[str]


def classify_node(state: RouterState) -> dict:
    raw = hybrid_classify(state["question"])
    intent = raw["intent"]
    confidence = raw.get("confidence", 0.9)
    if raw["source"] == "llm":
        intent = apply_confidence_threshold(IntentResult(
            intent=intent, confidence=confidence, reason=raw.get("reason", "")
        ))
    return {
        "intent": intent,
        "confidence": confidence,
        "source": raw["source"],
        "path": state.get("path", []) + [f"classify:{raw['source']}→{intent}"],
    }


def route_by_intent(state: RouterState) -> str:
    return state["intent"]


def make_branch(name: str, handler):
    def branch_fn(state: RouterState) -> dict:
        return {
            "answer": handler(state["question"]),
            "path": state["path"] + [name],
        }
    return branch_fn


router_builder = StateGraph(RouterState)
router_builder.add_node("classify", classify_node)
router_builder.add_node("tool", make_branch("tool", offline_tool_agent))
router_builder.add_node("chat", make_branch("chat", handle_chat))
router_builder.add_node("escalate", make_branch("escalate", handle_escalate))
router_builder.add_node("unknown", make_branch("unknown", clarify_unknown))
router_builder.add_node("block", make_branch("block", handle_block))

router_builder.add_edge(START, "classify")
router_builder.add_conditional_edges("classify", route_by_intent)
for branch in ("tool", "chat", "escalate", "unknown", "block"):
    router_builder.add_edge(branch, END)

router_graph = router_builder.compile()

for q in ["Where is ORD-1001?", "Thanks!", "I need a human", "help me"]:
    out = router_graph.invoke({
        "question": q, "intent": "", "confidence": 0.0,
        "source": "", "answer": "", "path": [],
    })
    print(f"Q: {q}")
    print(f"  path: {' → '.join(out['path'])}")
    print(f"  answer: {out['answer'][:70]}...\n")

---

## 10. Combining Intent Router + Tool Agent Graph

In production, the intent layer sits **above** the ReAct tool graph. Only `tool` intent invokes `ToolNode` + model loop — keeping tool menus small and costs down.

In [ ]:
def offline_model_for_tools(state: MessagesState) -> dict:
    has_tools = any(isinstance(m, ToolMessage) for m in state["messages"])
    if not has_tools:
        user = next(m.content for m in state["messages"] if isinstance(m, HumanMessage))
        calls = []
        if "ord" in user.lower():
            calls.append({"name": "lookup_order", "args": {"order_id": "ORD-1001"}, "id": "c1", "type": "tool_call"})
        if "return" in user.lower() or "refund" in user.lower():
            calls.append({"name": "search_policy", "args": {"query": "returns"}, "id": "c2", "type": "tool_call"})
        if not calls:
            calls = [{"name": "search_policy", "args": {"query": "shipping"}, "id": "c0", "type": "tool_call"}]
        return {"messages": [AIMessage(content="", tool_calls=calls)]}
    snippets = [str(m.content)[:60] for m in state["messages"] if isinstance(m, ToolMessage)]
    return {"messages": [AIMessage(content="Here is what I found: " + " | ".join(snippets))]}


def build_tool_agent_graph():
    b = StateGraph(MessagesState)
    b.add_node("model", offline_model_for_tools)
    b.add_node("tools", TOOL_NODE)
    b.add_edge(START, "model")
    b.add_conditional_edges("model", tools_condition)
    b.add_edge("tools", "model")
    return b.compile()


TOOL_AGENT = build_tool_agent_graph()


def full_assistant(user_message: str) -> dict[str, Any]:
    """Intent gate → branch handler or tool agent graph."""
    classification = classify_with_trace(user_message)
    intent = classification["intent"]

    if intent == "tool":
        agent_result = TOOL_AGENT.invoke({"messages": [HumanMessage(content=user_message)]})
        answer = agent_result["messages"][-1].content
        path = ["intent:tool", "tool_agent_graph"]
    else:
        answer = classification["answer"]
        path = [f"intent:{intent}"]

    return {
        "intent": intent,
        "path": path,
        "answer": answer,
        "classification": classification,
    }


for q in ["Where is ORD-1001 and what's the return policy?", "Thanks!", "help me"]:
    result = full_assistant(q)
    print(f"Q: {q}")
    print(f"  intent={result['intent']} path={result['path']}")
    print(f"  {str(result['answer'])[:80]}...\n")

---

## 11. Anti-Patterns

| Anti-pattern | Problem | Fix |
|--------------|---------|-----|
| Always call tools | Wastes tokens on "thanks" | Intent gate first |
| Silent failure on unknown | User confusion | Graceful fallback + clarify |
| 50-intent taxonomy | Classifier drift, maintenance | 4–6 intents max |
| No escalate path | Stuck frustrated users | Always offer human handoff |
| LLM for every message | Cost + latency | Keyword cascade first |
| No confidence threshold | Wrong tool calls | Downgrade low-confidence to unknown |

---

## 12. Optional — Live LLM Intent Classification

Set `USE_LIVE_LLM = True` in the setup cell to use OpenAI structured output for ambiguous messages.

In [ ]:
if USE_LIVE_LLM:
    try:
        r = live_llm_intent("I'm frustrated — my package still hasn't arrived")
        routed = apply_confidence_threshold(r)
        print(f"Raw: {r.intent} ({r.confidence:.2f}) → Routed: {routed}")
        print(f"Reason: {r.reason}")
    except Exception as exc:
        print("Live LLM failed:", exc)
else:
    print("Offline mode. Hybrid router uses keyword + offline_llm_intent.")
    demo = classify_with_trace("I'm not sure what I need")
    print(f"Demo: intent={demo['intent']} source={demo['source']}")

---

## 13. Check Your Understanding

1. Why route before invoking a tool-calling agent?
2. What is the hybrid cascade pattern (keyword → LLM)?
3. When should low confidence downgrade `tool` to `unknown`?
4. What makes fallback copy "graceful" vs robotic?
5. How does the LangGraph intent router differ from calling `dispatch()` in Python?
6. Where does the tool agent graph sit relative to intent detection?
7. Name two anti-patterns when designing intent taxonomies.

---

## 14. Summary

- **Intent detection** gates tool calls — route `tool`, `chat`, `escalate`, `unknown`, and `block` explicitly.
- Use **keyword classifiers** first (fast, free); **LLM structured output** for ambiguous cases.
- Apply **confidence thresholds** — low confidence → clarify, not wrong tool calls.
- **Graceful fallback** states what you can do and asks one clarifying question.
- **LangGraph conditional edges** implement the same routing as `if/elif` with visible graphs.
- Place the **intent layer above** the ReAct tool graph — only `tool` intent pays for tool loops.
- Log `intent`, `source`, and `confidence` on every request for observability.

Good intent routing saves cost, reduces errors, and keeps users oriented when the assistant cannot help.